# Huấn luyện các mô hình cho UEH Canteen Checkout trên Google Colab

Notebook này chỉ dành cho **Google Colab**. Ba mô hình được tổ chức thành ba phần độc lập và chỉ dùng chung phần chuẩn bị môi trường:

1. Bộ phân loại 11 món ăn.
2. Bộ phát hiện YOLO cho vùng thức ăn `food_region`.
3. Bộ phát hiện YOLO phụ trợ `egg`/`fish`.

Không dùng **Run all** nếu bạn chỉ muốn huấn luyện một mô hình. Hãy chạy toàn bộ **Phần 2**, sau đó chạy phần của mô hình cần huấn luyện.

## 1. Tổng quan quy trình và chuẩn bị dữ liệu trên máy local

### 1.1. Chiều đi của dữ liệu và kết quả

```text
Máy local
  data nguồn -> build/package -> outputs/cloud/*.zip
                              |
                              v
Google Drive Desktop -> MyDrive/canteen_checkout/datasets/
                              |
                              v
Google Colab -> train -> MyDrive/canteen_checkout/models/
                       -> MyDrive/canteen_checkout/runs/<model>/<thời_gian>/
                              |
                              v
Google Drive Desktop -> script pull -> models/ và outputs/cloud/drive_runs/
```

Quy tắc quan trọng:

- Dataset và notebook đi từ local lên Drive.
- Mô hình và báo cáo sau huấn luyện đi từ Drive về local.
- Không huấn luyện mô hình thành phẩm vào thư mục gốc của repo.
- Không dùng mô hình local cũ để ghi đè lên mô hình đã huấn luyện trên Colab.

### 1.2. Tạo, đóng gói và đưa dữ liệu đầu vào lên Drive

Chạy các lệnh sau trong PowerShell tại máy local trước khi mở Colab. Thư mục `data/archive/` chỉ là nguồn đọc và không bị sửa.

```powershell
# Bộ phân loại 11 món
./.venv/Scripts/python.exe scripts/data/01_build_classification_dataset.py --clear
./.venv/Scripts/python.exe scripts/data/03_package_classification_dataset.py

# Bộ phát hiện egg/fish có hard negative
./.venv/Scripts/python.exe scripts/data/04_build_yolo_dataset.py --clear
./.venv/Scripts/python.exe scripts/data/05_build_yolo_shared_negatives.py --clear
./.venv/Scripts/python.exe scripts/data/06_package_yolo_dataset.py `
  --source data/detection/egg_fish_shared `
  --output outputs/cloud/egg_fish_shared_yolo.zip `
  --manifest outputs/cloud/egg_fish_shared_yolo.manifest.json

# Bộ phát hiện vùng thức ăn
./.venv/Scripts/python.exe scripts/data/08_build_food_region_dataset.py --dry-run
./.venv/Scripts/python.exe scripts/data/08_build_food_region_dataset.py --clear
./.venv/Scripts/python.exe scripts/data/06_package_yolo_dataset.py `
  --source data/detection/food_regions `
  --output outputs/cloud/food_regions_yolo.zip `
  --manifest outputs/cloud/food_regions_yolo.manifest.json

# Chỉ đưa dataset và tệp dự án lên Drive, không đẩy mô hình local
./.venv/Scripts/python.exe scripts/cloud/01_sync_drive_artifacts.py --push-inputs --apply
```

Chờ Google Drive Desktop đồng bộ xong trước khi chạy Phần 2.

## 2. Chuẩn bị chung trên Google Colab

Phần này phải được chạy một lần cho mỗi phiên Colab. Phần này không huấn luyện mô hình.

### 2.1. Cấu hình repo và thư mục tạm

In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/Vo-Minh-Tri1412/cnn-food-recognition.git"
BRANCH = "main"

WORK_ROOT = Path("/content")
PROJECT_ROOT = WORK_ROOT / "cnn-food-recognition"
RUNTIME_DATA_ROOT = WORK_ROOT / "canteen_checkout_data"
YOLO_CACHE_ROOT = WORK_ROOT / "canteen_yolo_cache"

print("Thư mục làm việc:", WORK_ROOT)
print("Nhánh Git sẽ sử dụng:", BRANCH)

### 2.2. Clone repo và cài thư viện phụ thuộc

In [ ]:
def chay_lenh(command, cwd=None, env=None):
    command = [str(value) for value in command]
    print()
    print("Đang chạy:", " ".join(command))
    subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )


if not PROJECT_ROOT.exists():
    chay_lenh(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_ROOT])
else:
    chay_lenh(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
    chay_lenh(["git", "checkout", BRANCH], cwd=PROJECT_ROOT)
    chay_lenh(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_ROOT)

chay_lenh(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=PROJECT_ROOT,
)

print("Đã chuẩn bị mã nguồn tại:", PROJECT_ROOT)

### 2.3. Kiểm tra GPU và thư viện

In [ ]:
import torch
import ultralytics

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU CUDA. Vào Runtime > Change runtime type và chọn T4 GPU.")

print("GPU đang dùng:", torch.cuda.get_device_name(0))

### 2.4. Mount Google Drive và tạo cấu trúc lưu trữ

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/canteen_checkout")
DRIVE_DATASETS = DRIVE_ROOT / "datasets"
DRIVE_MODELS = DRIVE_ROOT / "models"
DRIVE_RUNS = DRIVE_ROOT / "runs"
DRIVE_PROJECT_FILES = DRIVE_ROOT / "project_files"

for folder in (DRIVE_DATASETS, DRIVE_MODELS, DRIVE_RUNS, DRIVE_PROJECT_FILES):
    folder.mkdir(parents=True, exist_ok=True)

print("Thư mục Drive gốc:", DRIVE_ROOT)
print("Dataset đầu vào:", DRIVE_DATASETS)
print("Mô hình thành phẩm:", DRIVE_MODELS)
print("Lần chạy và báo cáo:", DRIVE_RUNS)

### 2.5. Hàm dùng chung để kiểm tra checksum, giải nén và tạo môi trường huấn luyện

In [ ]:
from IPython.display import Image, display


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def lay_tep_drive(filename):
    path = DRIVE_DATASETS / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy {filename} tại {DRIVE_DATASETS}. "
            "Hãy build/package trên local rồi chạy --push-inputs --apply."
        )
    return path


def kiem_tra_manifest(zip_name, manifest_name):
    zip_path = lay_tep_drive(zip_name)
    manifest_path = lay_tep_drive(manifest_name)
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected = manifest.get("archive_sha256")
    actual = sha256_file(zip_path)
    if expected and actual.lower() != expected.lower():
        raise RuntimeError(f"Checksum không khớp cho {zip_name}: {actual} != {expected}")
    print(f"Checksum hợp lệ cho {zip_name}: {actual[:12]}...")
    return zip_path, manifest


def giai_nen_dataset(zip_name, manifest_name):
    zip_path, manifest = kiem_tra_manifest(zip_name, manifest_name)
    layout_root = manifest["layout_root"]
    target = RUNTIME_DATA_ROOT / layout_root
    if target.exists():
        shutil.rmtree(target)
    RUNTIME_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(RUNTIME_DATA_ROOT)
    if not target.exists():
        raise FileNotFoundError(f"Giải nén xong nhưng không thấy thư mục {target}")
    print("Dataset đã giải nén:", target)
    return target, manifest


def dem_anh(root):
    extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    root = Path(root)
    return sum(1 for path in root.rglob("*") if path.is_file() and path.suffix.lower() in extensions)


def tao_run(model_key):
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    run_root = DRIVE_RUNS / model_key / run_id
    run_root.mkdir(parents=True, exist_ok=True)
    return run_id, run_root


def tao_env_train(run_root):
    env = os.environ.copy()
    env["CANTEEN_MODEL_DIR"] = str(DRIVE_MODELS)
    env["CANTEEN_OUTPUTS_DIR"] = str(run_root)
    return env


def hien_thi_anh(paths, limit=8):
    for path in list(paths)[:limit]:
        print(path)
        display(Image(filename=str(path)))

### 2.6. Kiểm tra dữ liệu đầu vào trên Drive

In [ ]:
INPUT_ARTIFACTS = [
    "classification.zip",
    "classification.manifest.json",
    "food_regions_yolo.zip",
    "food_regions_yolo.manifest.json",
    "egg_fish_shared_yolo.zip",
    "egg_fish_shared_yolo.manifest.json",
]

missing = []
for filename in INPUT_ARTIFACTS:
    path = DRIVE_DATASETS / filename
    print(("Đã có" if path.exists() else "Còn thiếu"), "-", filename)
    if not path.exists():
        missing.append(filename)

if missing:
    raise FileNotFoundError("Drive còn thiếu dữ liệu đầu vào: " + ", ".join(missing))

print("Đã đủ dữ liệu đầu vào để chọn và huấn luyện từng mô hình.")

## 3. Mô hình phân loại 11 món ăn

Phần này chỉ phụ thuộc Phần 2. Candidate được fine-tune từ model production và lưu trong `Drive/runs/classifier/`; model production không bị ghi đè.

### 3.1. Cấu hình bộ phân loại

In [ ]:
CLS_ARCH = "efficientnet_b2"
CLS_EPOCHS = 8
CLS_PATIENCE = 3
CLS_BATCH = 16
CLS_IMAGE_SIZE = 260
CLS_LR = 1e-5
CLS_AUGMENTATION = "light"
CLS_LABEL_SMOOTHING = 0.02
CLS_LOSS = "cross_entropy"
CLS_SAMPLER = "shuffle"

CLS_RUN_ID, CLS_RUN_ROOT = tao_run("classifier")
CLS_INIT_CHECKPOINT = DRIVE_ROOT / "init_checkpoints" / "dish_classifier_production_20260620.pt"
CLS_MODEL_OUT = CLS_RUN_ROOT / "dish_classifier_candidate.pt"
CLS_CLASS_NAMES_OUT = CLS_RUN_ROOT / "class_names.json"
CLS_LOCAL_CLASS_NAMES = PROJECT_ROOT / "models" / "class_names.json"

print("Mã lần chạy bộ phân loại:", CLS_RUN_ID)
print("Checkpoint khởi tạo:", CLS_INIT_CHECKPOINT)
print("Candidate:", CLS_MODEL_OUT)
print("Thư mục báo cáo:", CLS_RUN_ROOT)

### 3.2. Giải nén và kiểm tra dataset phân loại

In [ ]:
CLS_DATA_ROOT, CLS_MANIFEST = giai_nen_dataset(
    "classification.zip",
    "classification.manifest.json",
)

for split in ("train", "val", "test"):
    print(f"{split}: {dem_anh(CLS_DATA_ROOT / split)} ảnh")

print(json.dumps(CLS_MANIFEST.get("class_totals", {}), indent=2, ensure_ascii=False))

### 3.3. Huấn luyện bộ phân loại và ghi mô hình trực tiếp vào Drive

In [ ]:
CLS_ENV = tao_env_train(CLS_RUN_ROOT)
CLS_COMMAND = [
    sys.executable,
    "scripts/train/01_train_classifier.py",
    "--data", CLS_DATA_ROOT,
    "--model-out", CLS_MODEL_OUT,
    "--init-checkpoint", CLS_INIT_CHECKPOINT,
    "--epochs", CLS_EPOCHS,
    "--patience", CLS_PATIENCE,
    "--batch-size", CLS_BATCH,
    "--image-size", CLS_IMAGE_SIZE,
    "--lr", CLS_LR,
    "--arch", CLS_ARCH,
    "--augmentation", CLS_AUGMENTATION,
    "--label-smoothing", CLS_LABEL_SMOOTHING,
    "--loss", CLS_LOSS,
    "--sampler", CLS_SAMPLER,
    "--no-weighted-loss",
]

chay_lenh(CLS_COMMAND, cwd=PROJECT_ROOT, env=CLS_ENV)

if not CLS_MODEL_OUT.exists() or not CLS_LOCAL_CLASS_NAMES.exists():
    raise FileNotFoundError("Huấn luyện kết thúc nhưng chưa đủ candidate và class_names.json.")
shutil.copy2(CLS_LOCAL_CLASS_NAMES, CLS_CLASS_NAMES_OUT)
print("Đã lưu classifier candidate:", CLS_MODEL_OUT)

### 3.4. Đọc báo cáo của bộ phân loại

In [ ]:
CLS_REPORTS = CLS_RUN_ROOT / "reports"
classification_report = CLS_REPORTS / "classification_report.txt"
if classification_report.exists():
    print(classification_report.read_text(encoding="utf-8"))
else:
    print("Chưa tìm thấy classification_report.txt")

hien_thi_anh(
    [path for path in (CLS_REPORTS / "training_history.png", CLS_REPORTS / "confusion_matrix.png") if path.exists()]
)

### 3.5. Tạo Grad-CAM cho bộ phân loại

In [ ]:
CLS_GRADCAM_ROOT = CLS_RUN_ROOT / "gradcam"
CLS_GRADCAM_COMMAND = [
    sys.executable,
    "scripts/debug/01_gradcam_debug.py",
    "--model", CLS_MODEL_OUT,
    "--data", CLS_DATA_ROOT / "test",
    "--out", CLS_GRADCAM_ROOT,
]
chay_lenh(CLS_GRADCAM_COMMAND, cwd=PROJECT_ROOT, env=CLS_ENV)
hien_thi_anh(sorted(CLS_GRADCAM_ROOT.rglob("*.png")))

## 4. Mô hình phát hiện vùng thức ăn `food_region`

Phần này chỉ phụ thuộc Phần 2. Mô hình nền mặc định là `yolo11s.pt`; trọng số tải sẵn và mô hình kiểm tra AMP được giữ trong `/content/canteen_yolo_cache`, không nằm trong repo.

### 4.1. Cấu hình bộ phát hiện vùng thức ăn

In [ ]:
REGION_BASE_MODEL = "yolo11s.pt"
REGION_EPOCHS = 100
REGION_IMAGE_SIZE = 640
REGION_BATCH = 16
REGION_PATIENCE = 20

REGION_RUN_ID, REGION_RUN_ROOT = tao_run("food_region")
REGION_MODEL_OUT = DRIVE_MODELS / "food_region_detector.pt"

print("Mã lần chạy vùng thức ăn:", REGION_RUN_ID)
print("Mô hình thành phẩm:", REGION_MODEL_OUT)
print("Điểm lưu và báo cáo:", REGION_RUN_ROOT)

### 4.2. Giải nén và kiểm tra dataset vùng thức ăn

In [ ]:
REGION_DATA_ROOT, REGION_MANIFEST = giai_nen_dataset(
    "food_regions_yolo.zip",
    "food_regions_yolo.manifest.json",
)
REGION_DATA_YAML = REGION_DATA_ROOT / "data.yaml"

if not REGION_DATA_YAML.exists():
    raise FileNotFoundError(f"Không tìm thấy {REGION_DATA_YAML}")

for split in ("train", "valid", "test"):
    print(f"{split}: {dem_anh(REGION_DATA_ROOT / split / 'images')} ảnh")
print(REGION_DATA_YAML.read_text(encoding="utf-8"))

### 4.3. Huấn luyện bộ phát hiện vùng thức ăn và ghi điểm lưu vào Drive

In [ ]:
REGION_ENV = tao_env_train(REGION_RUN_ROOT)
REGION_COMMAND = [
    sys.executable,
    "scripts/train/03_train_food_region_detector.py",
    "--data", REGION_DATA_YAML,
    "--model", REGION_BASE_MODEL,
    "--model-out", REGION_MODEL_OUT,
    "--project", REGION_RUN_ROOT / "yolo",
    "--name", "train",
    "--epochs", REGION_EPOCHS,
    "--imgsz", REGION_IMAGE_SIZE,
    "--batch", REGION_BATCH,
    "--patience", REGION_PATIENCE,
    "--weights-cache", YOLO_CACHE_ROOT,
]

chay_lenh(REGION_COMMAND, cwd=PROJECT_ROOT, env=REGION_ENV)

if not REGION_MODEL_OUT.exists():
    raise FileNotFoundError("Huấn luyện kết thúc nhưng chưa có food_region_detector.pt trên Drive.")
print("Đã lưu bộ phát hiện vùng thức ăn:", REGION_MODEL_OUT)

### 4.4. Đánh giá trên tập test

In [ ]:
from ultralytics import YOLO

REGION_TEST_METRICS = YOLO(str(REGION_MODEL_OUT)).val(
    data=str(REGION_DATA_YAML),
    split="test",
    imgsz=REGION_IMAGE_SIZE,
    batch=REGION_BATCH,
    project=str(REGION_RUN_ROOT / "yolo"),
    name="test",
)
print(json.dumps(REGION_TEST_METRICS.results_dict, indent=2, ensure_ascii=False))

### 4.5. Đọc báo cáo và biểu đồ vùng thức ăn

In [ ]:
REGION_SUMMARY = REGION_RUN_ROOT / "reports" / "food_region_detector_training_summary.json"
if REGION_SUMMARY.exists():
    print(REGION_SUMMARY.read_text(encoding="utf-8"))
else:
    print("Chưa tìm thấy food_region_detector_training_summary.json")

hien_thi_anh(sorted((REGION_RUN_ROOT / "yolo").rglob("*.png")))

## 5. Mô hình phát hiện phụ trợ `egg`/`fish`

Phần này chỉ phụ thuộc Phần 2 và dùng dataset có hard negative đã được chuẩn bị trên local.

### 5.1. Cấu hình bộ phát hiện egg/fish

In [ ]:
EGG_FISH_BASE_MODEL = "yolo11s.pt"
EGG_FISH_EPOCHS = 100
EGG_FISH_IMAGE_SIZE = 640
EGG_FISH_BATCH = 16
EGG_FISH_PATIENCE = 20

EGG_FISH_RUN_ID, EGG_FISH_RUN_ROOT = tao_run("egg_fish")
EGG_FISH_MODEL_OUT = DRIVE_MODELS / "egg_fish_detector.pt"

print("Mã lần chạy egg/fish:", EGG_FISH_RUN_ID)
print("Mô hình thành phẩm:", EGG_FISH_MODEL_OUT)
print("Điểm lưu và báo cáo:", EGG_FISH_RUN_ROOT)

### 5.2. Giải nén và kiểm tra dataset egg/fish

In [ ]:
EGG_FISH_DATA_ROOT, EGG_FISH_MANIFEST = giai_nen_dataset(
    "egg_fish_shared_yolo.zip",
    "egg_fish_shared_yolo.manifest.json",
)
EGG_FISH_DATA_YAML = EGG_FISH_DATA_ROOT / "data.yaml"

if not EGG_FISH_DATA_YAML.exists():
    raise FileNotFoundError(f"Không tìm thấy {EGG_FISH_DATA_YAML}")

for split in ("train", "valid", "test"):
    print(f"{split}: {dem_anh(EGG_FISH_DATA_ROOT / split / 'images')} ảnh")
print(EGG_FISH_DATA_YAML.read_text(encoding="utf-8"))

### 5.3. Huấn luyện bộ phát hiện egg/fish và ghi điểm lưu vào Drive

In [ ]:
EGG_FISH_ENV = tao_env_train(EGG_FISH_RUN_ROOT)
EGG_FISH_COMMAND = [
    sys.executable,
    "scripts/train/02_train_yolo_detector.py",
    "--data", EGG_FISH_DATA_YAML,
    "--model", EGG_FISH_BASE_MODEL,
    "--model-out", EGG_FISH_MODEL_OUT,
    "--project", EGG_FISH_RUN_ROOT / "yolo",
    "--name", "train",
    "--epochs", EGG_FISH_EPOCHS,
    "--imgsz", EGG_FISH_IMAGE_SIZE,
    "--batch", EGG_FISH_BATCH,
    "--patience", EGG_FISH_PATIENCE,
    "--weights-cache", YOLO_CACHE_ROOT,
]

chay_lenh(EGG_FISH_COMMAND, cwd=PROJECT_ROOT, env=EGG_FISH_ENV)

if not EGG_FISH_MODEL_OUT.exists():
    raise FileNotFoundError("Huấn luyện kết thúc nhưng chưa có egg_fish_detector.pt trên Drive.")
print("Đã lưu bộ phát hiện egg/fish:", EGG_FISH_MODEL_OUT)

### 5.4. Đánh giá trên tập test

In [ ]:
from ultralytics import YOLO

EGG_FISH_TEST_METRICS = YOLO(str(EGG_FISH_MODEL_OUT)).val(
    data=str(EGG_FISH_DATA_YAML),
    split="test",
    imgsz=EGG_FISH_IMAGE_SIZE,
    batch=EGG_FISH_BATCH,
    project=str(EGG_FISH_RUN_ROOT / "yolo"),
    name="test",
)
print(json.dumps(EGG_FISH_TEST_METRICS.results_dict, indent=2, ensure_ascii=False))

### 5.5. Đọc báo cáo và biểu đồ egg/fish

In [ ]:
EGG_FISH_SUMMARY = EGG_FISH_RUN_ROOT / "reports" / "egg_fish_detector_training_summary.json"
if EGG_FISH_SUMMARY.exists():
    print(EGG_FISH_SUMMARY.read_text(encoding="utf-8"))
else:
    print("Chưa tìm thấy egg_fish_detector_training_summary.json")

hien_thi_anh(sorted((EGG_FISH_RUN_ROOT / "yolo").rglob("*.png")))

## 6. Đưa mô hình và báo cáo về máy local bằng Google Drive Desktop

Sau khi huấn luyện xong mô hình cần thiết:

1. Kiểm tra các tệp thành phẩm trong `MyDrive/canteen_checkout/models/`.
2. Chờ Google Drive Desktop báo đồng bộ hoàn tất.
3. Chạy các lệnh sau tại repo trên máy local.

```powershell
# Xem trạng thái mô hình và các lần chạy trên Drive
./.venv/Scripts/python.exe scripts/cloud/01_sync_drive_artifacts.py --status

# Kéo mô hình chuẩn và lần chạy mới nhất của từng mô hình về project local
./.venv/Scripts/python.exe scripts/cloud/01_sync_drive_artifacts.py --pull-results --apply

# Kiểm tra mô hình local
Get-ChildItem models

# Chạy demo local
./.venv/Scripts/python.exe scripts/apps/01_demo_checkout_app.py --host 127.0.0.1 --port 7863
```

Các mô hình dùng khi chạy ứng dụng phải nằm tại:

```text
models/dish_classifier.pt
models/class_names.json
models/food_region_detector.pt
models/egg_fish_detector.pt
```